# AIC211 · Topic 2 · Tuần 6 — TV1 (Trí): Phòng thủ D2 + E2 theo cấu hình phòng thủ + bảng tổng hợp

**Mục tiêu:** có model D2 và D1+D2; biết gap đổi nguồn thay đổi thế nào khi bật phòng thủ; có bảng tổng hợp tự sinh từ `results.csv`.
**Cần có trước:** cả hai xong **tuần 5** (`thresholds.json`, `RF_F1.joblib`, `src/defenses.py`).

| Bước | Làm gì | Ước tính |
|---|---|---|
| 1–3 | Nạp dữ liệu, ghi `src/d2.py` | 5 phút |
| 4 | Train RF_F1D2 và RF_F1D2_D1 | 10–25 phút, chủ yếu chờ |
| 5 | Bảng E2 theo phòng thủ | 1 phút |
| 6 | Bảng tổng hợp cho báo cáo | 1 phút |
| 7 | Viết kết quả E1, E2 | Gửi số cho Claude viết nháp |

In [ ]:
# Ô 1 — Cài thư viện + nối Google Drive (chạy lại mỗi lần mở Colab)
import os, sys, importlib
try:
    import google.colab  # chỉ có khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install tldextract rapidfuzz shap xgboost
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/AIC211_URL'   # thư mục chung của nhóm
else:
    ROOT = os.path.abspath('AIC211_URL')          # chạy thử trên máy (không phải Colab)

for d in ['data/raw', 'data/processed', 'src', 'models', 'results', 'figures', 'notebooks']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
sys.path.insert(0, f'{ROOT}/src')
print('ROOT =', ROOT)
print(sorted(os.listdir(ROOT)))

In [ ]:
# Ô 2 — Nạp code, dữ liệu, model tuần 5
for f in ['models/thresholds.json', 'models/RF_F1.joblib', 'src/defenses.py']:
    assert os.path.exists(f'{ROOT}/{f}'), f'Thiếu {f}: chạy xong tuần 5 của cả hai trước.'
import common; importlib.reload(common)
from common import *
import json, joblib, random, time
import matplotlib.pyplot as plt
A = pd.read_parquet(f'{ROOT}/data/processed/A.parquet')
B = pd.read_parquet(f'{ROOT}/data/processed/B.parquet')
THR = json.load(open(f'{ROOT}/models/thresholds.json'))
INFO = json.load(open(f'{ROOT}/models/model_info.json'))
tr, va, te = (A.split == 'train').values, (A.split == 'val').values, (A.split == 'test').values
import defenses; importlib.reload(defenses)
from defenses import d1_normalize
print('A:', A.shape, '| B:', B.shape)

## Bước 3–4 — D2: feature theo cấu trúc
D2 tính feature **riêng** cho subdomain, domain, path, query và thêm khoảng cách tới tên thương hiệu (`paypa1` cách `paypal` 1 ký tự). Mục tiêu chống T2 (thêm subdomain), T5 (gạch nối), T6 (typo). Ngưỡng lưu vào `thresholds_D2.json`, không đụng `thresholds.json`.

In [ ]:
# Ô 3 — Ghi file src/d2.py: D2 = feature theo cấu trúc + khoảng cách tới tên thương hiệu
D2_CODE = r"""
import numpy as np, pandas as pd
from rapidfuzz.distance import Levenshtein
from common import split_parts, entropy, EXTRACT, make_F1

BRANDS = ['paypal', 'apple', 'microsoft', 'google', 'amazon', 'facebook', 'netflix', 'instagram',
          'whatsapp', 'chase', 'wellsfargo', 'bankofamerica', 'dhl', 'fedex', 'office365', 'outlook']

def d2_features(url_norm):
    host, rest = split_parts(url_norm)
    ext = EXTRACT(host); path, _, query = rest.partition('?')
    dom, sub = ext.domain, ext.subdomain
    dists = [Levenshtein.distance(dom, b) for b in BRANDS]
    j = int(np.argmin(dists))
    f = {'brand_min_dist': dists[j],
         'brand_typo': int(0 < dists[j] <= 2),          # gần giống thương hiệu nhưng KHÔNG phải thương hiệu
         'brand_in_sub': int(any(b in sub for b in BRANDS)),
         'brand_in_path': int(any(b in rest.lower() for b in BRANDS)),
         'n_sub_labels': len(sub.split('.')) if sub else 0}
    for name, part in [('sub', sub), ('dom', dom), ('path', path), ('query', query)]:
        f[f'len_{name}'] = len(part)
        f[f'digits_{name}'] = sum(c.isdigit() for c in part)
        f[f'hyph_{name}'] = part.count('-')
        f[f'ent_{name}'] = entropy(part)
    return {f'd2_{k}': v for k, v in f.items()}   # tiền tố d2_ để không trùng tên cột với F1

def make_F1D2(df):
    d2 = pd.DataFrame([d2_features(u) for u in df['url_norm']], index=df.index)
    return pd.concat([make_F1(df), d2], axis=1)
"""
with open(f'{ROOT}/src/d2.py', 'w', encoding='utf-8') as f:
    f.write(D2_CODE)
import d2; importlib.reload(d2)
from d2 import d2_features, make_F1D2
print(d2_features('paypa1-support.com/login'))

In [ ]:
# Ô 4 — Train RF trên F1 + D2 (RF_F1D2) và RF trên F1 + D2 có D1 (RF_F1D2_D1); ngưỡng FPR 1% trên val A
from sklearn.base import clone

def d1_frame(df):
    u = [normalize_url(d1_normalize('http://' + x)) for x in df['url_norm']]
    out = pd.DataFrame({'url_norm': u, 'label': df['label'].values})
    parts = out['url_norm'].map(split_parts)
    out['host'] = parts.str[0]; out['rest'] = parts.str[1]
    return out

rf_base = joblib.load(f'{ROOT}/models/RF_F1.joblib')
CONFIGS = {'RF_F1D2': (A, B), 'RF_F1D2_D1': (d1_frame(A), d1_frame(B))}
THR_D2, rows = {}, []
for name, (a, b) in CONFIGS.items():
    t0 = time.time()
    Xa, Xb = make_F1D2(a), make_F1D2(b)
    m = clone(rf_base).fit(Xa[tr], a.label[tr])
    THR_D2[name] = threshold_at_fpr(a.label[va], get_scores(m, Xa[va]))
    joblib.dump(m, f'{ROOT}/models/{name}.joblib')
    defense = 'D2' if name == 'RF_F1D2' else 'D1+D2'
    for src, X, y in [('A-test', Xa[te], a.label[te]), ('B', Xb, b.label)]:
        met = evaluate(y, get_scores(m, X), THR_D2[name])
        log_result(ROOT, 'E4_clean', 'RF_F1', src, met, defense=defense)
        rows.append({'cấu hình': defense, 'nguồn': src, **met})
    print(f'{name} xong ({time.time() - t0:.0f}s)')
json.dump(THR_D2, open(f'{ROOT}/models/thresholds_D2.json', 'w'), indent=2)   # file riêng, KHÔNG đụng thresholds.json
json.dump({**INFO, 'RF_F1D2': {'features': 'F1D2'}, 'RF_F1D2_D1': {'features': 'F1D2', 'preprocess': 'D1'}},
          open(f'{ROOT}/models/model_info.json', 'w'), indent=2)

In [ ]:
# Ô 5 — E2 theo cấu hình phòng thủ: gap đổi nguồn có hẹp lại không? (không phòng thủ vs D1 vs D2 vs D1+D2)
res = pd.read_csv(f'{ROOT}/results/results.csv')
base = res[(res.exp == 'E2') & (res.model == 'RF_F1')].assign(defense='none')
clean = res[(res.exp == 'E4_clean') & (res.model == 'RF_F1')]
tab = pd.concat([base, clean]).drop_duplicates(['defense', 'source'], keep='last')
f1 = tab.pivot(index='defense', columns='source', values='f1')
f1['gap_F1 (A−B)'] = f1['A-test'] - f1['B']
fpr = tab.pivot(index='defense', columns='source', values='fpr').add_prefix('FPR ')
print(pd.concat([f1, fpr], axis=1).round(4).to_string())
print('\n(D1 có sau khi Nhựt chạy Ô 7 notebook tuần 6. Chưa có thì chạy lại ô này sau.)')

In [ ]:
# Ô 6 — Hạ tầng tổng hợp: biến results.csv thành bảng cho báo cáo (mean ± std nếu có nhiều seed)
def summarize(exp, index, cols='seed', metric='f1'):
    d = res[res.exp == exp]
    if d.empty: return None
    g = d.groupby(index)[metric].agg(['mean', 'std', 'count'])
    return g.round(4)

res = pd.read_csv(f'{ROOT}/results/results.csv')
KEY = ['exp', 'model', 'defense', 'source', 'seed', 'k', 'ops', 'mode']
res = res.drop_duplicates(KEY, keep='last')     # chạy lại ô nhiều lần thì chỉ giữ lần mới nhất
os.makedirs(f'{ROOT}/results/tables', exist_ok=True)
specs = {'E1_split': (['source'], 'f1'), 'E1_drop': (['k', 'source'], 'f1'), 'E2': (['model', 'source'], 'f1'),
         'E3': (['model', 'source', 'mode', 'ops', 'k'], 'recall_attack'), 'E4_clean': (['defense', 'source'], 'f1')}
for exp, (idx, metric) in specs.items():
    t = summarize(exp, idx, metric=metric)
    if t is not None:
        t.to_csv(f'{ROOT}/results/tables/{exp}.csv')
        print(f'--- {exp} ({metric}) ---'); print(t.head(12).to_string())
print('\nBảng lưu ở results/tables/. Tuần 7 có 3 seed thì cột std tự có số.')

## Xong tuần 6 khi
- [ ] `src/d2.py`, model `RF_F1D2`, `RF_F1D2_D1`, `thresholds_D2.json` → **nhắn Nhựt** (tuần 7 Nhựt tấn công các model này)
- [ ] Bảng E2 theo phòng thủ (Ô 5), bảng tổng hợp trong `results/tables/`
- [ ] Gửi Claude: file `results.csv`, thư mục `results/tables/`, hình E1/E2 tuần 5 → viết phần kết quả E1, E2
- [ ] Thứ Năm họp với Nhựt: chọn hình chính

Tuần 7: chạy 3 seed cho E3, E4; 20 URL sạch bị chặn nhầm ở nguồn B; viết phần Security discussion.